# Toy Test: YOLOv8-pose-P6 Pruning End-to-End

P6 sibling of `toy_test.ipynb`. Runs the **P6 simple pruning script** on the tiny `coco8-pose` dataset (8 images, ~1–3 min on a single GPU) and verifies that:

1. Training completes — both C2 (backbone) and C2f (neck) blocks survive the surgery
2. The saved `last.pt` contains the actual pruned tensors (smaller params)
3. The `pruned_model.onnx` export reflects the real pruned architecture
4. Validation on the loaded model matches the auto-logged metrics

**Key differences from the non-P6 toy:**
- Uses `prune_yolov8_pose_p6_simple.py` (not the n-pose script)
- Model: `yolov8x-pose-p6.pt` (~190 MB download on first run; 99 M params)
- imgsz=960 (P6 minimum; smaller crashes the Detect head's Concat geometry)
- batch=2 (P6 has ~30× more params than n-pose, so batch must be small)
- Unpickling requires importing **both** `C2_v2` AND `C2f_v2` from the P6 module (P6 mixes both block types)

Wall time: ~1–3 min on RTX 5090 for the toy config.

## 1. Environment check

Same as the non-P6 toy — verify dependencies + CUDA. If `yolov8x-pose-p6.pt` isn't cached locally, the first run will download ~190 MB from the ultralytics CDN.

In [5]:
import torch
import ultralytics
import torch_pruning as tp
import fasterai

print(f'torch         {torch.__version__}')
print(f'ultralytics   {ultralytics.__version__}')
print(f'torch-pruning {tp.__version__ if hasattr(tp, "__version__") else "(no version attr)"}')
print(f'fasterai      {fasterai.__version__ if hasattr(fasterai, "__version__") else "(no version attr)"}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU mem total:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

torch         2.9.1+cu128
ultralytics   8.3.162
torch-pruning (no version attr)
fasterai      0.3.0
CUDA available: True
GPU mem total:  33.7 GB


## 2. Run the P6 pruning script

Tiny config: 2 epochs on coco8-pose at imgsz=960, batch=2, ratio=0.05 (gentle 5 % target compression for a toy run).

The script runs `replace_csp_blocks_in_place` first — this swaps **both** C2 blocks (backbone, ~6 of them in yolov8x-pose-p6) **and** C2f blocks (neck, ~6 of them) with their pruning-friendly `_v2` variants. The output line `Replaced CSP blocks: {'C2': N, 'C2f': M}` confirms both block types were found.

Wall time: ~1–3 min on RTX 5090. First run is slower because of the model download.

In [6]:
import subprocess
import sys

cmd = [
    sys.executable, 'prune_yolov8_pose_p6_simple.py',
    '--model', 'yolov8x-pose-p6.pt',
    '--data',  'coco8-pose.yaml',
    '--epochs', '2',
    '--imgsz',  '960',
    '--batch',  '2',
    '--lr',     '5e-5',
    '--ratio',  '0.05',
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print('--- tail of stdout ---')
print('\n'.join(result.stdout.splitlines()[-30:]))
if result.returncode != 0:
    print('--- stderr ---'); print(result.stderr[-2000:])
    raise RuntimeError(f'script exited with code {result.returncode}')

Running: /home/nathan/miniconda3/envs/dev/bin/python prune_yolov8_pose_p6_simple.py --model yolov8x-pose-p6.pt --data coco8-pose.yaml --epochs 2 --imgsz 960 --batch 2 --lr 5e-5 --ratio 0.05
--- tail of stdout ---
Starting training for 2 epochs...

[CB epoch 0 (pct_train=0.500)] Pruning step 1/2
   → 279.14 GMACs, 91.97M params (sparsity 7.3%, MACs −7.1%)

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
                   all          4         14      0.928      0.925      0.972      0.805       0.95      0.857      0.914      0.623

[Epoch 1/2] sparsity=7.3%  MACs−7.1%  prunes=1/2  mAP50-95_P=0.6234  mAP50_P=0.9137  mAP50-95_B=0.8053  mAP50_B=0.9716

[CB epoch 1 (pct_train=1.000)] Pruning step 2/2
   → 272.24 GMACs, 89.63M params (sparsity 9.6%, MACs −9.4%)

      Epoch    GPU_mem   box_loss  pose_loss  kobj_loss   cls_loss   dfl_loss  Instances       Size
                   all          4         14       0.98      0.857      0.964 

## 3. Inspect the auto-logged experiment

Same `experiments.jsonl` as the n-pose toy (the log is shared across all scripts). The last entry is our P6 run.

In [7]:
import json
from pathlib import Path

lines = Path('experiments.jsonl').read_text().strip().split('\n')
last_run = json.loads(lines[-1])

print(f'Label:           {last_run["label"]}')
print(f'Model:           {last_run["config"]["model"]}')
print(f'Wall time:       {last_run["wall_time_s"]:.1f} s')
print()
print(f'Baseline params: {last_run["baseline"]["params_m"]:.3f} M  (yolov8x-pose-p6 ≈ 99 M)')
print(f'Final    params: {last_run["final"]["params_m"]:.3f} M')
print(f'Param reduction: {last_run["final"]["param_reduction_pct"]:.2f} %')
print(f'MAC   reduction: {last_run["final"]["mac_reduction_pct"]:.2f} %')
print()
print(f'pose mAP50-95:   {last_run["final"]["pose_map5095"]:.4f}')
print(f'box  mAP50-95:   {last_run["final"]["box_map5095"]:.4f}')

Label:           2step_2ep_r0.05
Model:           yolov8x-pose-p6.pt
Wall time:       11.5 s

Baseline params: 99.183 M  (yolov8x-pose-p6 ≈ 99 M)
Final    params: 89.591 M
Param reduction: 9.67 %
MAC   reduction: 9.43 %

pose mAP50-95:   0.5923
box  mAP50-95:   0.8089


## 4. Verify `last.pt` IS the pruned P6 model

**Important P6 difference**: the unpickler needs **BOTH** `C2_v2` and `C2f_v2` from the P6 script (n-pose only has `C2f_v2`). Importing from the wrong module gives the wrong class definitions and unpickling fails.

In [8]:
import sys
sys.path.insert(0, '.')
# P6-specific: import BOTH block-replacement classes from the P6 module
from prune_yolov8_pose_p6 import C2_v2, C2f_v2
from ultralytics import YOLO

# Find the most recent train directory
train_dirs = sorted(Path('runs/pose').glob('train*'), key=lambda p: p.stat().st_mtime)
latest_pt = train_dirs[-1] / 'weights' / 'last.pt'
print(f'Loading: {latest_pt}')

yolo_loaded = YOLO(str(latest_pt))
loaded_params = sum(p.numel() for p in yolo_loaded.model.parameters()) / 1e6
print(f'Loaded model params: {loaded_params:.3f} M')
print(f'Match with log:      {"YES" if abs(loaded_params - last_run["final"]["params_m"]) < 0.01 else "NO"}')

# Confirm the surgery worked: count both block types in the loaded model
n_c2_v2 = sum(1 for m in yolo_loaded.model.modules() if type(m).__name__ == 'C2_v2')
n_c2f_v2 = sum(1 for m in yolo_loaded.model.modules() if type(m).__name__ == 'C2f_v2')
print(f'\nC2_v2 blocks:  {n_c2_v2}  (backbone — P6-specific)')
print(f'C2f_v2 blocks: {n_c2f_v2}  (neck — shared with non-P6 variants)')
if n_c2_v2 == 0:
    print('WARNING: no C2_v2 blocks found — surgery may have skipped them.')

Loading: runs/pose/train5/weights/last.pt
Loaded model params: 89.631 M
Match with log:      NO

C2_v2 blocks:  6  (backbone — P6-specific)
C2f_v2 blocks: 5  (neck — shared with non-P6 variants)


## 5. Compare conv filter shapes: baseline vs pruned

Load the fresh baseline `yolov8x-pose-p6.pt` and compare per-layer weight tensor shapes side-by-side with the pruned model from cell 4. This is the most direct evidence that pruning actually changed the architecture: protected layers (stem, keypoint head) should show **unchanged**; non-protected layers should have smaller output and/or input channel counts.

Note: paths like `model.<N>.cv1.conv` change shape semantics after the C2→C2_v2 surgery (one 2c-output conv → two c-output convs), so we only compare paths that are **structurally preserved** by the surgery — plain `Conv` layers (stem, downsamplers) and the `cv2` output-projection of each CSP block (the surgery reuses `cv2` unchanged).

In [9]:
from ultralytics import YOLO as YOLO_base

print('Loading fresh baseline yolov8x-pose-p6.pt for shape comparison...')
yolo_base = YOLO_base('yolov8x-pose-p6.pt')
base_params = sum(p.numel() for p in yolo_base.model.parameters()) / 1e6
print(f'Baseline params: {base_params:.3f} M  (vs pruned: {loaded_params:.3f} M)\n')

# Candidate layers — paths preserved across the C2→C2_v2 / C2f→C2f_v2 surgery.
# Mix of: protected stem, backbone downsamples, CSP output projections,
# and (if present) keypoint head conv.
candidates = [
    ('model.0.conv',         'stem (3-ch input — PROTECTED)'),
    ('model.1.conv',         'first backbone downsample'),
    ('model.3.conv',         'backbone downsample'),
    ('model.5.conv',         'backbone downsample'),
    ('model.7.conv',         'deeper backbone downsample'),
    ('model.9.conv',         'deepest backbone Conv (if present)'),
    ('model.2.cv2.conv',     'first CSP block — output projection'),
    ('model.4.cv2.conv',     'CSP block — output projection'),
    ('model.6.cv2.conv',     'CSP block — output projection'),
    ('model.8.cv2.conv',     'CSP block — output projection'),
]

base_modules  = dict(yolo_base.model.named_modules())
prune_modules = dict(yolo_loaded.model.named_modules())

print(f'{"Layer":<24} {"Baseline shape":<22} {"Pruned shape":<22} {"Δ":<22}  Description')
print('-' * 120)
for path, desc in candidates:
    b = base_modules.get(path); p = prune_modules.get(path)
    if b is None or p is None or not hasattr(b, 'weight') or not hasattr(p, 'weight'):
        continue
    bs, ps = tuple(b.weight.shape), tuple(p.weight.shape)
    if bs == ps:
        change = 'unchanged'
    else:
        parts = []
        if bs[0] != ps[0]: parts.append(f'out {bs[0]}→{ps[0]} (−{(1-ps[0]/bs[0])*100:.1f}%)')
        if bs[1] != ps[1]: parts.append(f'in {bs[1]}→{ps[1]} (−{(1-ps[1]/bs[1])*100:.1f}%)')
        change = ', '.join(parts) if parts else 'unchanged'
    print(f'{path:<24} {str(bs):<22} {str(ps):<22} {change:<22}  {desc}')

# Free the baseline model — we don't need it after the comparison
del yolo_base

Loading fresh baseline yolov8x-pose-p6.pt for shape comparison...
Baseline params: 99.183 M  (vs pruned: 89.631 M)

Layer                    Baseline shape         Pruned shape           Δ                       Description
------------------------------------------------------------------------------------------------------------------------
model.0.conv             (80, 3, 3, 3)          (80, 3, 3, 3)          unchanged               stem (3-ch input — PROTECTED)
model.1.conv             (160, 80, 3, 3)        (160, 80, 3, 3)        unchanged               first backbone downsample
model.3.conv             (320, 160, 3, 3)       (304, 152, 3, 3)       out 320→304 (−5.0%), in 160→152 (−5.0%)  backbone downsample
model.5.conv             (640, 320, 3, 3)       (608, 304, 3, 3)       out 640→608 (−5.0%), in 320→304 (−5.0%)  backbone downsample
model.7.conv             (640, 640, 3, 3)       (608, 608, 3, 3)       out 640→608 (−5.0%), in 640→608 (−5.0%)  deeper backbone downsample
model.9